# Watermark a PyTorch model and detect it

The owner creates an identity and a secret seed, embeds the watermark while training, and checks it on the quantised vector that a zero-knowledge proof would attest to. See `../../docs/protocol.md`.

In [ ]:
import gzip, numpy as np
from pathlib import Path
DATA = Path("../../demo/data")          # MNIST shipped with the repository
def idx(name):
    raw = gzip.open(DATA / name, "rb").read(); nd = raw[3]
    dims = [int.from_bytes(raw[4 + 4*i: 8 + 4*i], "big") for i in range(nd)]
    return np.frombuffer(raw, dtype=np.uint8, offset=4 + 4*nd).reshape(dims)
x = ((idx("train-images-idx3-ubyte.gz")[:12000] / 255.0 - 0.1307) / 0.3081).astype("float32")
y = idx("train-labels-idx1-ubyte.gz")[:12000].astype("int64")
xt = ((idx("t10k-images-idx3-ubyte.gz")[:2000] / 255.0 - 0.1307) / 0.3081).astype("float32")
yt = idx("t10k-labels-idx1-ubyte.gz")[:2000].astype("int64")
print(x.shape, xt.shape)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from sigil import protocol as P
from sigil.identity import KeyPair
from sigil.embed import embed_torch

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1, self.conv2 = nn.Conv2d(1, 32, 3), nn.Conv2d(32, 64, 3)      # conv2: M = 32*3*3 = 288
        self.fc1, self.fc2 = nn.Linear(9216, 128), nn.Linear(128, 10)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv2(F.relu(self.conv1(x)))), 2)
        return self.fc2(F.relu(self.fc1(torch.flatten(x, 1))))

owner = KeyPair.generate()
cfg = P.Config(T=64, M=288, layer="conv2")
seed, pkh = P.new_seed(), P.pk_hash(owner.public_bytes)
torch.manual_seed(0)
model, det = embed_torch(Net(), (torch.from_numpy(x).unsqueeze(1), torch.from_numpy(y)), cfg, seed, pkh, epochs=1)
print(det)

In [ ]:
with torch.no_grad():
    dev = next(model.parameters()).device
    acc = (model(torch.from_numpy(xt).unsqueeze(1).to(dev)).argmax(1).cpu().numpy() == yt).mean()
print(f"test accuracy {acc:.3f}")
torch.save(model.state_dict(), "watermarked.pt")
from sigil.adapters import detect
print("from file:", detect("watermarked.pt", seed, pkh, cfg))
print("someone else's key:", detect("watermarked.pt", P.new_seed(), pkh, cfg))
assert det.accepted